In [ ]:
import os
import glob
import utils
import utils
import importlib
import xarray as xr
import numpy as np

importlib.reload(utils)

print("starting script", flush=True)
base_path = "../data/rsds_past"

files_past = []
files_future = []

for root, dirs, files in os.walk(base_path):
    if not dirs:
        rel_path = os.path.relpath(root, base_path)
        path_past = os.path.join(base_path, rel_path)
        future_path = path_past.replace("rsds_past", "rsds_future")

        # Find nc files
        past_files = glob.glob(os.path.join(path_past, "*r1i1p1_1995*.nc"))
        future_files = glob.glob(os.path.join(future_path, "*r1i1p1_2045*.nc"))

        files_past.extend(past_files)
        files_future.extend(future_files)


# Use the file names as model identifiers
model_names_solar = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_past
]
model_names_future = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_future
]

print("File source loaded", flush=True)
# Process and select time frame
files_raw_past = utils.pre_process(files_past, ["rsds"], 0, 5, 0, 5)
files_solar = utils.select_time_frame(files_raw_past, slice("1995-01-01", "2004-12-30"))

print("Files loaded", flush=True)

print("starting script", flush=True)
base_path = "../data/wind_past"

files_past = []
files_future = []

for root, dirs, files in os.walk(base_path):
    if not dirs:
        rel_path = os.path.relpath(root, base_path)
        path_past = os.path.join(base_path, rel_path)
        future_path = path_past.replace("wind_past", "wind_future")

        # Find nc files
        past_files = glob.glob(os.path.join(path_past, "*r1i1p1_1995*.nc"))
        future_files = glob.glob(os.path.join(future_path, "*r1i1p1_2045*.nc"))

        files_past.extend(past_files)
        files_future.extend(future_files)


# Use the file names as model identifiers
model_names_wind = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_past
]
model_names_future = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_future
]

print("File source loaded", flush=True)
# Process and select time frame
files_raw_past = utils.pre_process(files_past, ["sfcWind"], 0, 5, 0, 5)
files_wind = utils.select_time_frame(files_raw_past, slice("1995-01-01", "2004-12-30"))

print("Files loaded", flush=True)

In [ ]:
marginal = utils.correlation_over_space(
    files_wind,
    files_solar,
    model_names_wind,
    model_names_solar,
)


seasons = ["DJF", "MAM", "JJA", "SON"]
# stack all models
model_arrays = []
for m in range(len(marginal)):
    season_dict = marginal[m]
    print(season_dict)
    combined = xr.concat(
        [season_dict[s] for s in seasons], dim=xr.Variable("season", seasons)
    )
    model_arrays.append(combined)

# concatenate along  model dimension
all_models_array = xr.concat(
    model_arrays, dim=xr.Variable("model", range(len(marginal)))
)

seasonal_mean = all_models_array.values
np.save(
    "../plotting_data/combined_spatial_raw/correlation.npy",
    seasonal_mean,
)

In [ ]:
dunkelflaute = utils.dunkelflauten_index(
    files_wind,
    files_solar,
    model_names_wind,
    model_names_solar,
)

seasons = ["DJF", "MAM", "JJA", "SON"]
# stack all models
model_arrays = []
for m in range(len(dunkelflaute)):
    season_dict = dunkelflaute[m]
    combined = xr.concat(
        [season_dict[s] for s in seasons], dim=xr.Variable("season", seasons)
    )
    model_arrays.append(combined)

# concatenate along  model dimension
all_models_array = xr.concat(
    model_arrays, dim=xr.Variable("model", range(len(marginal)))
)

dunkelflauten = all_models_array.values
np.save(
    "../plotting_data/combined_spatial_raw/dunkelflauten.npy",
    dunkelflauten,
)